# Dota 2 Analytics API: полный пример запросов

Ноутбук дергает все публичные endpoints локального FastAPI-сервиса, обновляет игрока, ждет job и выводит статистику в таблицы.

Перед запуском убедись, что API поднят, например:

```powershell
$env:DATABASE_URL='sqlite+pysqlite:///./dev.sqlite'
python -m uvicorn app.main:app --host 127.0.0.1 --port 8000
```

Swagger: http://127.0.0.1:8000/docs

In [1]:
from __future__ import annotations

import json
import os
import time
from datetime import datetime
from pathlib import Path
from typing import Any

import httpx
from dotenv import load_dotenv
from IPython.display import JSON, Markdown, display

try:
    import pandas as pd
except ImportError:
    pd = None

load_dotenv()

API_BASE = os.getenv("API_BASE", "http://127.0.0.1:8000").rstrip("/")
ACCOUNT_ID = int(os.getenv("ACCOUNT_ID", "175966938"))

API_BASE, ACCOUNT_ID

('http://127.0.0.1:8000', 175966938)

In [2]:
def api_request(
    method: str,
    path: str,
    *,
    params: dict[str, Any] | None = None,
    json_body: dict[str, Any] | None = None,
    allow_error: bool = False,
    timeout: float = 30.0,
) -> tuple[int, Any]:
    """Call local FastAPI service and return (status_code, json_or_text)."""
    url = f"{API_BASE}{path}"
    with httpx.Client(timeout=timeout) as client:
        response = client.request(method, url, params=params, json=json_body)

    try:
        payload = response.json()
    except ValueError:
        payload = response.text

    if not allow_error:
        response.raise_for_status()
    return response.status_code, payload


def as_table(data: Any, *, max_rows: int = 100):
    if pd is None:
        display(JSON(data))
        return None
    if isinstance(data, list):
        frame = pd.json_normalize(data)
    elif isinstance(data, dict):
        frame = pd.json_normalize(data)
    else:
        display(data)
        return None
    display(frame.head(max_rows))
    return frame


def show_json(title: str, data: Any):
    display(Markdown(f"## {title}"))
    if isinstance(data, dict | list):
        display(JSON(data, expanded=False))
        return
    text = str(data).replace("```", "'''")
    display(Markdown(f"```text\n{text}\n```"))


def show_table(title: str, data: Any, *, max_rows: int = 100):
    display(Markdown(f"## {title}"))
    return as_table(data, max_rows=max_rows)


def duration_label(seconds: int | None) -> str | None:
    if seconds is None:
        return None
    minutes, secs = divmod(int(seconds), 60)
    return f"{minutes}:{secs:02d}"


def kda(kills: int, deaths: int, assists: int) -> float:
    return round((kills + assists) / max(1, deaths), 2)

## 1. Healthcheck

In [3]:
status, health = api_request("GET", "/health", allow_error=True)
show_json(f"GET /health -> {status}", health)

if status != 200:
    raise RuntimeError(f"API is not healthy or not running at {API_BASE}: {health}")

## GET /health -> 200

<IPython.core.display.JSON object>

## 2. Каталог ачивок

`GET /v1/achievements` возвращает все определения ачивок без привязки к игроку.

In [4]:
status, achievement_catalog = api_request("GET", "/v1/achievements")
catalog_df = show_table(f"GET /v1/achievements -> {status}", achievement_catalog, max_rows=200)

if pd is not None:
    display(Markdown("### Ачивки по категориям"))
    display(catalog_df.groupby(["category", "tier"]).size().reset_index(name="count"))

## GET /v1/achievements -> 200

,id,title,description,category,tier,progress,target,progressPercent,unlocked,unavailable,unlockedAt
0,marathon_day,Marathon Day,Play 5+ games in one day.,activity,silver,0.0,5.0,0.0,False,False,None
1,returning_player,Returning Player,Play after a 14+ day break.,activity,bronze,0.0,1.0,0.0,False,False,None
2,weekly_grinder,Weekly Grinder,Play 10+ games in 7 days.,activity,bronze,0.0,10.0,0.0,False,False,None
3,chaos_enjoyer,Chaos Enjoyer,Play 3 games in a row lasting 45+ minutes.,fun,bronze,0.0,3.0,0.0,False,False,None
4,glass_cannon,Glass Cannon,Deal 35k+ hero damage with 10+ deaths.,fun,bronze,0.0,1.0,0.0,False,False,None
5,alchemist_chemical_salary,Забрал пенсию у бабки,Выиграй 10 игр подряд на Alchemist.,hero_flavor,silver,0.0,10.0,0.0,False,False,None
6,anti_mage_afk_forest,"АФК лес, потом GG проебали",Выиграй 10 игр подряд на Anti-Mage.,hero_flavor,gold,0.0,10.0,0.0,False,False,None
7,arc_warden_two_accounts,Два аккаунта в одной игре,Выиграй 10 игр подряд на Arc Warden.,hero_flavor,gold,0.0,10.0,0.0,False,False,None
8,axe_spin_doctor,Вертушка судьбы,Выиграй 10 игр подряд на Axe.,hero_flavor,silver,0.0,10.0,0.0,False,False,None
9,broodmother_real_estate,Риэлтор паутины,Выиграй 10 игр подряд на Broodmother.,hero_flavor,gold,0.0,10.0,0.0,False,False,None


### Ачивки по категориям

,category,tier,count
0,activity,bronze,2
1,activity,silver,1
2,fun,bronze,2
3,hero_flavor,gold,15
4,hero_flavor,legendary,1
5,hero_flavor,silver,13
6,hero_mastery,bronze,1
7,hero_mastery,gold,2
8,hero_mastery,silver,2
9,performance,bronze,1


## 3. Refresh игрока

`POST /v1/players/{account_id}/refresh` ставит фоновую job. Если сработал cooldown, ноутбук продолжит читать уже сохраненные данные.

In [5]:
refresh_path = f"/v1/players/{ACCOUNT_ID}/refresh"
status, refresh_response = api_request("POST", refresh_path, allow_error=True, timeout=60)
show_json(f"POST {refresh_path} -> {status}", refresh_response)

job = None
if status == 202:
    job = refresh_response
elif status == 429:
    display(Markdown(f"Cooldown: `{refresh_response.get('detail', refresh_response)}`. Используем сохраненную статистику."))
else:
    raise RuntimeError(f"Refresh failed: HTTP {status}: {refresh_response}")

job

## POST /v1/players/175966938/refresh -> 202

<IPython.core.display.JSON object>

{'id': 'cdc2fc1b-1d9d-4403-a1eb-b085f306d75a',
 'accountId': 175966938,
 'status': 'queued',
 'message': None,
 'createdAt': '2026-07-09T20:45:24',
 'startedAt': None,
 'finishedAt': None}

In [6]:
job_status_history = []

if job:
    job_id = job["id"]
    job_path = f"/v1/jobs/{job_id}"
    for _ in range(90):
        status, current_job = api_request("GET", job_path, allow_error=True)
        job_status_history.append(current_job)
        if current_job.get("status") in {"done", "failed"}:
            break
        time.sleep(2)

    show_table(f"GET {job_path} polling", job_status_history)
    if job_status_history[-1].get("status") != "done":
        raise RuntimeError(f"Refresh job did not finish successfully: {job_status_history[-1]}")
else:
    display(Markdown("Refresh job не создавалась из-за cooldown."))

## GET /v1/jobs/cdc2fc1b-1d9d-4403-a1eb-b085f306d75a polling

,id,accountId,status,message,createdAt,startedAt,finishedAt
0,cdc2fc1b-1d9d-4403-a1eb-b085f306d75a,175966938,failed,STRATZ GraphQL errors: [{'message': 'You have ...,2026-07-09T20:45:24,2026-07-09T20:45:24.397848,2026-07-09T20:45:25.899345


RuntimeError: Refresh job did not finish successfully: {'id': 'cdc2fc1b-1d9d-4403-a1eb-b085f306d75a', 'accountId': 175966938, 'status': 'failed', 'message': "STRATZ GraphQL errors: [{'message': 'You have surpassed the maximum take value of :  100'}]", 'createdAt': '2026-07-09T20:45:24', 'startedAt': '2026-07-09T20:45:24.397848', 'finishedAt': '2026-07-09T20:45:25.899345'}

## 4. Забираем все player endpoints одним блоком

In [7]:
endpoint_specs = {
    "summary": ("GET", f"/v1/players/{ACCOUNT_ID}/summary", None),
    "breakdowns": ("GET", f"/v1/players/{ACCOUNT_ID}/breakdowns", None),
    "matches_20": ("GET", f"/v1/players/{ACCOUNT_ID}/matches", {"limit": 20}),
    "matches_100": ("GET", f"/v1/players/{ACCOUNT_ID}/matches", {"limit": 100}),
    "heroes": ("GET", f"/v1/players/{ACCOUNT_ID}/heroes", None),
    "hero_global_winrates": ("GET", "/v1/heroes/winrates", {"limit": 50, "min_matches": 10000, "order_by": "winRate"}),
    "hero_builds_ogre_magi": ("GET", "/v1/heroes/84/builds", {"limit": 10, "min_matches": 1000, "match_limit": 10000}),
    "achievements_all": ("GET", f"/v1/players/{ACCOUNT_ID}/achievements", None),
    "achievements_unlocked": ("GET", f"/v1/players/{ACCOUNT_ID}/achievements", {"unlocked": "true"}),
    "achievements_locked": ("GET", f"/v1/players/{ACCOUNT_ID}/achievements", {"unlocked": "false"}),
    "rating": ("GET", f"/v1/players/{ACCOUNT_ID}/rating", None),
    "style_100": ("GET", f"/v1/players/{ACCOUNT_ID}/style", {"limit": 100}),
    "leaderboard": ("GET", "/v1/leaderboard", {"metric": "overall", "limit": 50}),
}

api_snapshot: dict[str, Any] = {
    "generatedAt": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "apiBase": API_BASE,
    "accountId": ACCOUNT_ID,
    "health": health,
    "achievement_catalog": achievement_catalog,
    "refresh": refresh_response,
    "jobHistory": job_status_history,
    "responses": {},
}

endpoint_statuses = []
for name, (method, path, params) in endpoint_specs.items():
    status, payload = api_request(method, path, params=params, allow_error=True)
    api_snapshot["responses"][name] = payload
    endpoint_statuses.append({
        "name": name,
        "method": method,
        "path": path,
        "params": params or {},
        "status": status,
        "items": len(payload) if isinstance(payload, list) else 1,
    })

show_table("Статусы всех API endpoints", endpoint_statuses)

C:\Users\Богдан\AppData\Local\Temp\ipykernel_39912\1414671534.py:18: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generatedAt": datetime.utcnow().isoformat(timespec="seconds") + "Z",


## Статусы всех API endpoints

,name,method,path,status,items,params.limit,params.min_matches,params.order_by,params.match_limit,params.unlocked,params.metric
0,summary,GET,/v1/players/175966938/summary,200,1,NaN,NaN,NaN,NaN,NaN,NaN
1,breakdowns,GET,/v1/players/175966938/breakdowns,200,1,NaN,NaN,NaN,NaN,NaN,NaN
2,matches_20,GET,/v1/players/175966938/matches,200,20,20.0,NaN,NaN,NaN,NaN,NaN
3,matches_100,GET,/v1/players/175966938/matches,200,100,100.0,NaN,NaN,NaN,NaN,NaN
4,heroes,GET,/v1/players/175966938/heroes,200,47,NaN,NaN,NaN,NaN,NaN,NaN
5,hero_global_winrates,GET,/v1/heroes/winrates,200,50,50.0,10000.0,winRate,NaN,NaN,NaN
6,hero_builds_ogre_magi,GET,/v1/heroes/84/builds,200,1,10.0,1000.0,NaN,10000.0,NaN,NaN
7,achievements_all,GET,/v1/players/175966938/achievements,200,59,NaN,NaN,NaN,NaN,NaN,NaN
8,achievements_unlocked,GET,/v1/players/175966938/achievements,200,21,NaN,NaN,NaN,NaN,true,NaN
9,achievements_locked,GET,/v1/players/175966938/achievements,200,38,NaN,NaN,NaN,NaN,false,NaN


,name,method,path,status,items,params.limit,params.min_matches,params.order_by,params.match_limit,params.unlocked,params.metric
0,summary,GET,/v1/players/175966938/summary,200,1,NaN,NaN,NaN,NaN,NaN,NaN
1,breakdowns,GET,/v1/players/175966938/breakdowns,200,1,NaN,NaN,NaN,NaN,NaN,NaN
2,matches_20,GET,/v1/players/175966938/matches,200,20,20.0,NaN,NaN,NaN,NaN,NaN
3,matches_100,GET,/v1/players/175966938/matches,200,100,100.0,NaN,NaN,NaN,NaN,NaN
4,heroes,GET,/v1/players/175966938/heroes,200,47,NaN,NaN,NaN,NaN,NaN,NaN
5,hero_global_winrates,GET,/v1/heroes/winrates,200,50,50.0,10000.0,winRate,NaN,NaN,NaN
6,hero_builds_ogre_magi,GET,/v1/heroes/84/builds,200,1,10.0,1000.0,NaN,10000.0,NaN,NaN
7,achievements_all,GET,/v1/players/175966938/achievements,200,59,NaN,NaN,NaN,NaN,NaN,NaN
8,achievements_unlocked,GET,/v1/players/175966938/achievements,200,21,NaN,NaN,NaN,NaN,true,NaN
9,achievements_locked,GET,/v1/players/175966938/achievements,200,38,NaN,NaN,NaN,NaN,false,NaN


## 5. Summary: профиль, winrate, KDA, карточка

In [8]:
summary = api_snapshot["responses"]["summary"]

summary_core = {
    key: value
    for key, value in summary.items()
    if key not in {"card", "avatar"}
}
show_table("GET /summary: основные поля", summary_core)

card = summary["card"]
card_rows = [
    {"metric": row["label"], "score": row["value"], "overall": card["overall"], "position": card["position"]}
    for row in card["rows"]
]
show_table("Карточка IMP/FRM/FGT/SUR/OBJ/UTL", card_rows)

show_table("Source метрики карточки", card.get("source", {}))

## GET /summary: основные поля

,accountId,name,rankTier,leaderboardRank,lastRefreshedAt,matches,wins,losses,winRate,kills,deaths,assists,kda,recentForm,breakdowns.byGameMode,breakdowns.byRanked
0,175966938,АДМИРАЛ,63,None,2026-07-09T11:52:21.338336,101,55,46,54.5,745,874,1309,2.35,"[L, L, W, W, W, W, W, W, L, W]","[{'gameMode': 23, 'label': 'Turbo', 'matches':...","[{'bucket': 'ranked', 'label': 'Ranked', 'matc..."


## Карточка IMP/FRM/FGT/SUR/OBJ/UTL

,metric,score,overall,position
0,IMP,73,75,OFF
1,FRM,92,75,OFF
2,FGT,83,75,OFF
3,SUR,59,75,OFF
4,OBJ,77,75,OFF
5,UTL,79,75,OFF


## Source метрики карточки

,winrate,recentWinrate,avgKills,avgDeaths,avgAssists,avgGpm,avgXpm,avgHeroDamage,avgTowerDamage,avgHealing,avgLastHits,roleWeights,limit
0,54.5,70.0,7.4,8.7,13.0,927.7,1571.9,26428,2992,865,158.0,OFF,500


,winrate,recentWinrate,avgKills,avgDeaths,avgAssists,avgGpm,avgXpm,avgHeroDamage,avgTowerDamage,avgHealing,avgLastHits,roleWeights,limit
0,54.5,70.0,7.4,8.7,13.0,927.7,1571.9,26428,2992,865,158.0,OFF,500


## 6. Breakdowns: режимы матчей и ranked/unranked

In [9]:
breakdowns = api_snapshot["responses"]["breakdowns"]

if pd is not None:
    show_table("GET /breakdowns: byGameMode", breakdowns.get("byGameMode", []), max_rows=50)
    show_table("GET /breakdowns: byRanked", breakdowns.get("byRanked", []), max_rows=10)
else:
    show_json("GET /breakdowns", breakdowns)

## GET /breakdowns: byGameMode

,gameMode,label,matches,wins,losses,winRate,kills,deaths,assists,kda,avgKills,avgDeaths,avgAssists
0,23,Turbo,65,38,27,58.5,452,514,738,2.32,7.0,7.9,11.4
1,22,Ranked All Pick,33,16,17,48.5,264,333,536,2.40,8.0,10.1,16.2
2,4,Single Draft,2,1,1,50.0,18,15,24,2.80,9.0,7.5,12.0
3,2,Captains Mode,1,0,1,0.0,11,12,11,1.83,11.0,12.0,11.0


## GET /breakdowns: byRanked

,bucket,label,matches,wins,losses,winRate,kills,deaths,assists,kda,avgKills,avgDeaths,avgAssists
0,ranked,Ranked,33,16,17,48.5,264,333,536,2.40,8.0,10.1,16.2
1,unranked,Unranked,68,39,29,57.4,481,541,773,2.32,7.1,8.0,11.4


## 7. Matches: последние матчи и агрегаты

In [ ]:
matches = api_snapshot["responses"]["matches_100"]

if pd is not None:
    matches_df = pd.DataFrame(matches)
    matches_df["duration"] = matches_df["durationSeconds"].map(duration_label)
    matches_df["kda"] = matches_df.apply(lambda row: kda(row.kills, row.deaths, row.assists), axis=1)
    matches_df["result"] = matches_df["won"].map({True: "WIN", False: "LOSS"})
    show_table("GET /matches?limit=100", matches_df)

    display(Markdown("### Агрегаты по матчам"))
    match_aggregates = {
        "matches": len(matches_df),
        "wins": int(matches_df["won"].sum()),
        "losses": int((~matches_df["won"]).sum()),
        "winRate": round(matches_df["won"].mean() * 100, 1),
        "avgKills": round(matches_df["kills"].mean(), 2),
        "avgDeaths": round(matches_df["deaths"].mean(), 2),
        "avgAssists": round(matches_df["assists"].mean(), 2),
        "avgKda": round(matches_df["kda"].mean(), 2),
        "avgGpm": round(matches_df["gpm"].mean(), 1),
        "avgXpm": round(matches_df["xpm"].mean(), 1),
        "avgHeroDamage": round(matches_df["heroDamage"].mean(), 0),
        "avgTowerDamage": round(matches_df["towerDamage"].mean(), 0),
        "avgHealing": round(matches_df["heroHealing"].mean(), 0),
        "maxKills": int(matches_df["kills"].max()),
        "maxGpm": int(matches_df["gpm"].max()),
        "maxHeroDamage": int(matches_df["heroDamage"].max()),
    }
    display(pd.DataFrame([match_aggregates]))
else:
    show_json("GET /matches?limit=100", matches)

## GET /matches?limit=100

,matchId,startedAt,durationSeconds,heroId,gameMode,gameModeName,lobbyType,rankedBucket,rankedLabel,won,...,gpm,xpm,heroDamage,towerDamage,heroHealing,lastHits,laneRole,duration,kda,result
0,8886131605,2026-07-07T21:28:23,2483,60,22,Ranked All Pick,7.0,ranked,Ranked,False,...,336,581,11064,26,0,120,None,41:23,0.92,LOSS
1,8886077137,2026-07-07T20:34:47,2579,84,22,Ranked All Pick,7.0,ranked,Ranked,True,...,650,762,23528,2944,0,189,None,42:59,2.42,WIN
2,8885808627,2026-07-07T17:03:21,1895,63,23,Turbo,0.0,unranked,Unranked,True,...,1807,2731,84610,11998,0,230,None,31:35,2.38,WIN
3,8885724269,2026-07-07T16:00:22,1191,99,23,Turbo,0.0,unranked,Unranked,True,...,1136,1476,13224,5370,0,116,None,19:51,7.00,WIN
4,8885652481,2026-07-07T15:12:21,2011,36,23,Turbo,0.0,unranked,Unranked,True,...,1747,2869,78860,3948,8289,240,None,33:31,2.64,WIN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,8636890026,2026-01-05T19:03:18,1407,126,23,Turbo,0.0,unranked,Unranked,False,...,887,1211,21322,357,0,89,None,23:27,1.50,LOSS
96,8636850122,2026-01-05T18:30:45,1372,30,23,Turbo,0.0,unranked,Unranked,True,...,670,1840,15006,464,5692,5,None,22:52,2.22,WIN
97,8636767166,2026-01-05T17:23:49,1351,79,23,Turbo,0.0,unranked,Unranked,True,...,1054,2106,18082,3424,0,74,None,22:31,3.50,WIN
98,8636725471,2026-01-05T16:51:04,1730,126,23,Turbo,0.0,unranked,Unranked,True,...,1283,2124,22750,6008,0,165,None,28:50,4.50,WIN


### Агрегаты по матчам

,matches,wins,losses,winRate,avgKills,avgDeaths,avgAssists,avgKda,avgGpm,avgXpm,avgHeroDamage,avgTowerDamage,avgHealing,maxKills,maxGpm,maxHeroDamage
0,100,55,45,55.0,7.39,8.69,13.03,3.11,927.1,1570.1,26459.0,3015.0,874.0,18,1873,84610


## 8. Heroes: пул героев, winrate, KDA

In [10]:
heroes = api_snapshot["responses"]["heroes"]

if pd is not None:
    heroes_df = pd.DataFrame(heroes)
    heroes_df = heroes_df.sort_values(["games", "winRate", "kda"], ascending=[False, False, False])
    show_table("GET /heroes", heroes_df, max_rows=200)

    display(Markdown("### Топ героев по играм"))
    display(heroes_df[["heroId", "games", "wins", "losses", "winRate", "kda", "avgGpm", "avgXpm", "recentForm"]].head(15))

    display(Markdown("### Лучший winrate среди героев с 3+ играми"))
    display(heroes_df[heroes_df["games"] >= 3].sort_values(["winRate", "games"], ascending=[False, False]).head(15))
else:
    show_json("GET /heroes", heroes)

## GET /heroes

,heroId,games,wins,losses,winRate,avgKills,avgDeaths,avgAssists,kda,avgGpm,avgXpm,recentForm
0,60,17,8,9,47.1,9.2,8.1,13.3,2.80,657.1,1013.1,"[L, W, W, W, W]"
1,84,7,5,2,71.4,7.3,8.1,13.1,2.51,991.6,1655.1,"[W, W, L, L, W]"
2,126,6,1,5,16.7,7.2,9.0,6.8,1.56,940.3,1641.8,"[L, L, L, L, L]"
3,2,4,2,2,50.0,8.2,9.8,11.5,2.03,967.2,1657.8,"[W, L, L, W]"
4,101,3,3,0,100.0,12.7,10.0,11.3,2.40,1145.3,2056.0,"[W, W, W]"
8,28,3,2,1,66.7,7.7,8.3,19.7,3.28,893.0,1527.3,"[W, L, W]"
5,8,3,2,1,66.7,5.3,4.3,6.0,2.62,1342.0,2307.3,"[W, L, W]"
7,1,3,2,1,66.7,7.7,6.0,7.0,2.44,1312.3,2312.7,"[L, W, W]"
6,31,3,2,1,66.7,7.0,12.3,16.3,1.89,572.7,1081.3,"[W, L, W]"
10,36,3,1,2,33.3,12.3,10.7,12.0,2.28,977.7,1587.7,"[W, L, L]"


### Топ героев по играм

,heroId,games,wins,losses,winRate,kda,avgGpm,avgXpm,recentForm
0,60,17,8,9,47.1,2.80,657.1,1013.1,"[L, W, W, W, W]"
1,84,7,5,2,71.4,2.51,991.6,1655.1,"[W, W, L, L, W]"
2,126,6,1,5,16.7,1.56,940.3,1641.8,"[L, L, L, L, L]"
3,2,4,2,2,50.0,2.03,967.2,1657.8,"[W, L, L, W]"
4,101,3,3,0,100.0,2.40,1145.3,2056.0,"[W, W, W]"
8,28,3,2,1,66.7,3.28,893.0,1527.3,"[W, L, W]"
5,8,3,2,1,66.7,2.62,1342.0,2307.3,"[W, L, W]"
7,1,3,2,1,66.7,2.44,1312.3,2312.7,"[L, W, W]"
6,31,3,2,1,66.7,1.89,572.7,1081.3,"[W, L, W]"
10,36,3,1,2,33.3,2.28,977.7,1587.7,"[W, L, L]"


### Лучший winrate среди героев с 3+ играми

,heroId,games,wins,losses,winRate,avgKills,avgDeaths,avgAssists,kda,avgGpm,avgXpm,recentForm
4,101,3,3,0,100.0,12.7,10.0,11.3,2.40,1145.3,2056.0,"[W, W, W]"
1,84,7,5,2,71.4,7.3,8.1,13.1,2.51,991.6,1655.1,"[W, W, L, L, W]"
8,28,3,2,1,66.7,7.7,8.3,19.7,3.28,893.0,1527.3,"[W, L, W]"
5,8,3,2,1,66.7,5.3,4.3,6.0,2.62,1342.0,2307.3,"[W, L, W]"
7,1,3,2,1,66.7,7.7,6.0,7.0,2.44,1312.3,2312.7,"[L, W, W]"
6,31,3,2,1,66.7,7.0,12.3,16.3,1.89,572.7,1081.3,"[W, L, W]"
3,2,4,2,2,50.0,8.2,9.8,11.5,2.03,967.2,1657.8,"[W, L, L, W]"
0,60,17,8,9,47.1,9.2,8.1,13.3,2.80,657.1,1013.1,"[L, W, W, W, W]"
10,36,3,1,2,33.3,12.3,10.7,12.0,2.28,977.7,1587.7,"[W, L, L]"
9,99,3,1,2,33.3,4.7,9.7,11.3,1.66,838.3,1188.0,"[W, L, L]"


## 9. Global Heroes: общий STRATZ winrate

In [11]:
hero_global_winrates = api_snapshot["responses"]["hero_global_winrates"]

if pd is not None:
    hero_global_df = pd.DataFrame(hero_global_winrates)
    show_table("GET /v1/heroes/winrates", hero_global_df, max_rows=100)
else:
    show_json("GET /v1/heroes/winrates", hero_global_winrates)

## GET /v1/heroes/winrates

,heroId,name,displayName,shortName,imageUrl,roles,matchCount,winCount,lossCount,winRate
0,67,npc_dota_hero_spectre,Spectre,spectre,None,"[CARRY, ESCAPE, DURABLE]",155204,87256,67948,56.22
1,94,npc_dota_hero_medusa,Medusa,medusa,None,"[CARRY, DURABLE, DISABLER]",49930,27405,22525,54.89
2,12,npc_dota_hero_phantom_lancer,Phantom Lancer,phantom_lancer,None,"[CARRY, ESCAPE, NUKER, PUSHER]",131865,72110,59755,54.68
3,42,npc_dota_hero_skeleton_king,Wraith King,skeleton_king,None,"[CARRY, INITIATOR, DURABLE, DISABLER, SUPPORT]",154556,84066,70490,54.39
4,44,npc_dota_hero_phantom_assassin,Phantom Assassin,phantom_assassin,None,"[CARRY, ESCAPE]",184479,99381,85098,53.87
5,54,npc_dota_hero_life_stealer,Lifestealer,life_stealer,None,"[CARRY, ESCAPE, DURABLE, DISABLER]",126645,67832,58813,53.56
6,113,npc_dota_hero_arc_warden,Arc Warden,arc_warden,None,"[CARRY, ESCAPE, NUKER]",45330,24177,21153,53.34
7,20,npc_dota_hero_vengefulspirit,Vengeful Spirit,vengefulspirit,None,"[ESCAPE, NUKER, INITIATOR, DISABLER, SUPPORT]",176176,93739,82437,53.21
8,41,npc_dota_hero_faceless_void,Faceless Void,faceless_void,None,"[CARRY, ESCAPE, INITIATOR, DURABLE, DISABLER]",139705,74259,65446,53.15
9,80,npc_dota_hero_lone_druid,Lone Druid,lone_druid,None,"[CARRY, DURABLE, PUSHER]",24659,13055,11604,52.94


## 10. Hero Builds: метовые сборки STRATZ

In [12]:
hero_builds = api_snapshot["responses"]["hero_builds_ogre_magi"]

show_json("GET /v1/heroes/84/builds", {k: v for k, v in hero_builds.items() if k not in {"coreItems", "startingItems", "boots", "neutralItems", "talents", "skillBuild", "guides"}})

if pd is not None:
    for section in ["coreItems", "startingItems", "boots", "neutralItems", "talents", "skillBuild", "guides"]:
        display(Markdown(f"### {section}"))
        display(pd.json_normalize(hero_builds.get(section) or []).head(30))
else:
    show_json("Hero builds sections", hero_builds)

## GET /v1/heroes/84/builds

<IPython.core.display.JSON object>

### coreItems

,itemId,itemName,shortName,matchCount,winCount,winRate,metaScore,averageTimeMinute,averageTimeSeconds,wasGiven,equippedMatchCount,equippedWinCount
0,65,Hand of Midas,hand_of_midas,182532,96396,52.81,83.48,16.3,None,None,None,None
1,88,Ring of Basilius,ring_of_basilius,150303,77897,51.83,71.66,5.3,None,None,None,None
2,36,Magic Wand,magic_wand,76131,39359,51.70,45.21,2.4,None,None,None,None
3,73,Bracer,bracer,28793,14564,50.58,27.96,1.5,None,None,None,None
4,178,Soul Ring,soul_ring,10022,5062,50.51,21.25,2.0,None,None,None,None


### startingItems

...

,abilityId,abilityName,matchCount,winCount,winRate,metaScore,averageTimeMinute,averageTimeSeconds
0,403,+12 Ignite DPS,229615,117917,51.35,82.97,18.9,1134.3
1,6183,+35 Bloodlust Attack Speed,143614,79992,55.70,60.15,30.2,1810.0
2,442,+80 Damage,109300,59470,54.41,49.98,28.7,1723.6
3,6708,+30 Strength,99284,59847,60.28,49.20,36.9,2213.0
4,6707,+220 Fireblast Damage,74264,43999,59.25,41.76,36.7,2203.2
5,1181,+2/0.01 Dumb Luck Mana/Mana Regen Per Strength,55019,28219,51.29,33.53,19.2,1153.8
6,7030,17% Fireblast chance on attack,25937,16476,63.52,29.57,45.0,2698.3
7,730,special_bonus_attributes,20391,11003,53.96,24.66,24.6,1473.5


### skillBuild

,level,abilityId,abilityName,matchCount,winCount,winRate,metaScore
0,1,5439,Ignite,253610,129794,51.18,76.02
1,2,5438,Fireblast,230674,118386,51.32,70.81
2,3,5440,Bloodlust,21935,10876,49.58,22.38
3,4,5440,Bloodlust,90504,46555,51.44,38.74
4,5,5440,Bloodlust,9321,4632,49.69,19.53
5,6,5441,Multicast,283696,144835,51.05,82.87
6,7,5440,Bloodlust,15158,7705,50.83,21.26
7,8,5440,Bloodlust,27344,13902,50.84,24.06
8,9,5440,Bloodlust,15685,8109,51.70,21.69
9,10,403,+12 Ignite DPS,99304,50668,51.02,40.61


### guides

,matchId,createdAt,itemIds,itemNames,neutralItemIds,neutralItemNames
0,8888708218,2026-07-09T19:47:02Z,[],[],[],[]
1,8888704788,2026-07-09T19:25:21Z,[],[],[],[]
2,8888630763,2026-07-09T18:34:28Z,[],[],[],[]
3,8888714232,2026-07-09T18:31:36Z,[],[],[],[]
4,8888637197,2026-07-09T18:30:43Z,[],[],[],[]
5,8888679610,2026-07-09T18:28:39Z,[],[],[],[]
6,8888616710,2026-07-09T18:26:06Z,[],[],[],[]
7,8888585822,2026-07-09T18:06:32Z,[],[],[],[]
8,8888596466,2026-07-09T17:56:35Z,[],[],[],[]
9,8888589789,2026-07-09T17:39:26Z,[],[],[],[]


## 11. Achievements: открытые, закрытые, прогресс

In [13]:
achievements = api_snapshot["responses"]["achievements_all"]
unlocked = api_snapshot["responses"]["achievements_unlocked"]
locked = api_snapshot["responses"]["achievements_locked"]

if pd is not None:
    achievement_columns = [
        "id",
        "title",
        "category",
        "tier",
        "progress",
        "target",
        "progressPercent",
        "unlocked",
        "unavailable",
        "evidence",
    ]

    def achievement_frame(items):
        rows = []
        for item in items:
            row = dict(item)
            row["evidence"] = json.dumps(row.get("evidence") or {}, ensure_ascii=False)
            rows.append(row)
        return pd.DataFrame(rows).reindex(columns=achievement_columns)

    achievements_df = achievement_frame(achievements)
    show_table("GET /achievements: все", achievements_df, max_rows=200)

    display(Markdown("### Открытые ачивки"))
    unlocked_df = achievement_frame(unlocked).sort_values(["category", "tier", "title"])
    display(unlocked_df[["id", "title", "category", "tier", "progress", "target", "progressPercent", "evidence"]].head(200))

    display(Markdown("### Ближайшие закрытые ачивки"))
    locked_df = achievement_frame(locked).sort_values("progressPercent", ascending=False)
    display(locked_df[["id", "title", "category", "tier", "progress", "target", "progressPercent", "unavailable", "evidence"]].head(25))

    display(Markdown("### Ачивки по категориям"))
    display(
        achievements_df.groupby(["category", "unlocked"])
        .size()
        .reset_index(name="count")
        .sort_values(["category", "unlocked"])
    )
else:
    show_json("GET /achievements", achievements)

## GET /achievements: все

,id,title,category,tier,progress,target,progressPercent,unlocked,unavailable,evidence
0,returning_player,Returning Player,activity,bronze,1.000000,1.0,100.0,True,False,"{""matchId"": 8669297557, ""gapDays"": 23, ""limit""..."
1,weekly_grinder,Weekly Grinder,activity,bronze,23.000000,10.0,100.0,True,False,"{""limit"": 500}"
2,marathon_day,Marathon Day,activity,silver,8.000000,5.0,100.0,True,False,"{""limit"": 500}"
3,chaos_enjoyer,Chaos Enjoyer,fun,bronze,6.000000,3.0,100.0,True,False,"{""limit"": 500}"
4,glass_cannon,Glass Cannon,fun,bronze,1.000000,1.0,100.0,True,False,"{""limit"": 500}"
5,anti_mage_afk_forest,"АФК лес, потом GG проебали",hero_flavor,gold,2.000000,10.0,20.0,False,False,"{""heroId"": 1, ""heroName"": ""Anti-Mage"", ""matchI..."
6,arc_warden_two_accounts,Два аккаунта в одной игре,hero_flavor,gold,0.000000,10.0,0.0,False,False,"{""heroId"": 113, ""heroName"": ""Arc Warden"", ""mat..."
7,broodmother_real_estate,Риэлтор паутины,hero_flavor,gold,0.000000,10.0,0.0,False,False,"{""heroId"": 61, ""heroName"": ""Broodmother"", ""mat..."
8,chen_zoo_director,Директор зоопарка,hero_flavor,gold,0.000000,5.0,0.0,False,False,"{""heroId"": 66, ""heroName"": ""Chen"", ""matchIds"":..."
9,clinkz_tower_arson,Поджог недвижимости,hero_flavor,gold,0.000000,10000.0,0.0,False,False,"{""heroId"": 56, ""heroName"": ""Clinkz"", ""matchIds..."


### Открытые ачивки

,id,title,category,tier,progress,target,progressPercent,evidence
0,returning_player,Returning Player,activity,bronze,1.0,1.0,100.0,"{""matchId"": 8669297557, ""gapDays"": 23, ""limit""..."
1,weekly_grinder,Weekly Grinder,activity,bronze,23.0,10.0,100.0,"{""limit"": 500}"
2,marathon_day,Marathon Day,activity,silver,8.0,5.0,100.0,"{""limit"": 500}"
3,chaos_enjoyer,Chaos Enjoyer,fun,bronze,6.0,3.0,100.0,"{""limit"": 500}"
4,glass_cannon,Glass Cannon,fun,bronze,1.0,1.0,100.0,"{""limit"": 500}"
5,ogre_magi_ludik_ebanny,Лудик ебанный,hero_flavor,gold,42525.0,30000.0,100.0,"{""heroId"": 84, ""heroName"": ""Ogre Magi"", ""match..."
6,cursed_breaker,Cursed Breaker,hero_mastery,gold,1.0,1.0,100.0,"{""matchId"": 8848789080, ""heroId"": 60, ""limit"":..."
7,wide_pool,Wide Pool,hero_mastery,silver,35.0,10.0,100.0,"{""heroIds"": [1, 2, 3, 8, 10, 25, 28, 29, 30, 3..."
8,kda_5,KDA Machine,performance,bronze,11.0,5.0,100.0,"{""matchId"": 8848789080, ""limit"": 500}"
9,clean_game,Clean Game,performance,gold,1.0,1.0,100.0,"{""matchId"": 8848789080, ""deaths"": 2, ""killPart..."


### Ближайшие закрытые ачивки

,id,title,category,tier,progress,target,progressPercent,unavailable,evidence
36,support_brain,Support Brain,role,silver,0.990000,1.0,99.0,False,"{""position"": ""OFF"", ""score"": 79, ""limit"": 500}"
34,mid_pressure,Mid Pressure,role,silver,0.990000,1.0,99.0,False,"{""position"": ""OFF"", ""score"": 83, ""limit"": 500}"
33,carry_core,Carry Core,role,silver,0.990000,1.0,99.0,False,"{""position"": ""OFF"", ""score"": 92, ""limit"": 500}"
35,offlane_impact,Offlane Impact,role,silver,0.880000,1.0,88.0,False,"{""position"": ""OFF"", ""score"": 66.0, ""limit"": 500}"
29,signature_hero,Signature Hero,hero_mastery,gold,0.784314,1.0,78.4,False,"{""heroId"": 60, ""games"": 17, ""wins"": 8, ""winRat..."
30,hero_loyalist,Hero Loyalist,hero_mastery,silver,17.000000,25.0,68.0,False,"{""limit"": 500}"
32,flex_player,Flex Player,role,gold,2.000000,3.0,66.7,False,"{""roles"": [""MID"", ""OFF""], ""limit"": 500}"
37,hot_streak_10,Hot Streak X,win_form,gold,6.000000,10.0,60.0,False,"{""limit"": 500}"
28,hero_spammer,Hero Spammer,hero_mastery,bronze,5.000000,10.0,50.0,False,"{""limit"": 500}"
0,anti_mage_afk_forest,"АФК лес, потом GG проебали",hero_flavor,gold,2.000000,10.0,20.0,False,"{""heroId"": 1, ""heroName"": ""Anti-Mage"", ""matchI..."


### Ачивки по категориям

,category,unlocked,count
0,activity,True,3
1,fun,True,2
2,hero_flavor,False,28
3,hero_flavor,True,1
4,hero_mastery,False,3
5,hero_mastery,True,2
6,performance,False,1
7,performance,True,9
8,role,False,5
9,win_form,False,1


## 12. Achievements filters: категории и unlocked

In [14]:
categories = sorted({item["category"] for item in achievement_catalog})
category_responses = []

for category in categories:
    status, payload = api_request(
        "GET",
        f"/v1/players/{ACCOUNT_ID}/achievements",
        params={"category": category},
        allow_error=True,
    )
    category_responses.append({
        "category": category,
        "status": status,
        "items": len(payload) if isinstance(payload, list) else 0,
        "unlocked": sum(1 for item in payload if isinstance(payload, list) and item.get("unlocked")),
        "response": payload,
    })

show_table("GET /achievements?category=...", [{k: v for k, v in row.items() if k != "response"} for row in category_responses])

## GET /achievements?category=...

,category,status,items,unlocked
0,activity,200,3,3
1,fun,200,2,2
2,hero_flavor,200,29,1
3,hero_mastery,200,5,2
4,performance,200,10,9
5,role,200,5,0
6,win_form,200,5,4


,category,status,items,unlocked
0,activity,200,3,3
1,fun,200,2,2
2,hero_flavor,200,29,1
3,hero_mastery,200,5,2
4,performance,200,10,9
5,role,200,5,0
6,win_form,200,5,4


## 13. Rating: overall, role, breakdown

In [15]:
rating = api_snapshot["responses"]["rating"]
show_table("GET /rating: snapshot", {k: v for k, v in rating.items() if k not in {"breakdown", "source"}})

breakdown = rating.get("breakdown", {})
if isinstance(breakdown, dict):
    rows = breakdown.get("rows", breakdown)
else:
    rows = breakdown
show_table("Rating breakdown", rows)
show_table("Rating source", rating.get("source", {}))

## GET /rating: snapshot

,accountId,overall,position,createdAt
0,175966938,75,OFF,2026-07-09T20:45:54.919845Z


## Rating breakdown

,label,value
0,IMP,73
1,FRM,92
2,FGT,83
3,SUR,59
4,OBJ,77
5,UTL,79


## Rating source

,winrate,recentWinrate,avgKills,avgDeaths,avgAssists,avgGpm,avgXpm,avgHeroDamage,avgTowerDamage,avgHealing,avgLastHits,roleWeights,limit
0,54.5,70.0,7.4,8.7,13.0,927.7,1571.9,26428,2992,865,158.0,OFF,500


,winrate,recentWinrate,avgKills,avgDeaths,avgAssists,avgGpm,avgXpm,avgHeroDamage,avgTowerDamage,avgHealing,avgLastHits,roleWeights,limit
0,54.5,70.0,7.4,8.7,13.0,927.7,1571.9,26428,2992,865,158.0,OFF,500


## 14. Очки стиля за последние 100 матчей

In [ ]:
style_100 = api_snapshot["responses"]["style_100"]

if not isinstance(style_100, dict):
    show_json("GET /style?limit=100", style_100)
else:
    style_matches = pd.DataFrame(style_100.get("matches") or [])
    if style_matches.empty:
        display(Markdown("Нет матчей для расчета style score."))
    else:
        style_matches["score"] = pd.to_numeric(style_matches["score"], errors="coerce")
        style_matches["itemScore"] = pd.to_numeric(style_matches["itemScore"], errors="coerce")
        style_matches["skillScore"] = pd.to_numeric(style_matches["skillScore"], errors="coerce")
        rated_style = style_matches.dropna(subset=["score"]).copy()
        overview = pd.DataFrame([
            {
                "requestedMatches": style_100.get("matchesConsidered"),
                "ratedMatches": len(rated_style),
                "apiAverageStyle": style_100.get("styleScore"),
                "notebookAverageStyle": round(rated_style["score"].mean(), 1) if not rated_style.empty else None,
                "averageItems": round(rated_style["itemScore"].mean(), 1) if not rated_style.empty else None,
                "averageSkills": round(rated_style["skillScore"].mean(), 1) if not rated_style.empty else None,
                "label": style_100.get("label"),
            }
        ])
        show_table("Средний style score за 100 матчей", overview)

        by_hero = (
            rated_style.groupby(["heroId", "heroName"], dropna=False, as_index=False)
            .agg(
                matches=("matchId", "count"),
                averageStyle=("score", "mean"),
                averageItems=("itemScore", "mean"),
                averageSkills=("skillScore", "mean"),
                maxStyle=("score", "max"),
                minStyle=("score", "min"),
            )
            .sort_values(["averageStyle", "matches"], ascending=[False, False])
        )
        for column in ["averageStyle", "averageItems", "averageSkills", "maxStyle", "minStyle"]:
            by_hero[column] = by_hero[column].round(1)
        show_table("Style score по героям", by_hero, max_rows=200)
        display(Markdown("### Самые нестандартные матчи"))
        display(rated_style.sort_values("score", ascending=False)[["matchId", "heroId", "heroName", "score", "itemScore", "skillScore"]].head(20))

## 15. Leaderboard

In [ ]:
leaderboard = api_snapshot["responses"]["leaderboard"]
show_table("GET /v1/leaderboard?metric=overall&limit=50", leaderboard, max_rows=100)

## 15. Полный snapshot всех ответов

Переменная `api_snapshot` содержит все ответы. Следующая ячейка показывает верхний уровень и сохраняет JSON рядом с ноутбуком.

In [ ]:
snapshot_path = Path(f"api_snapshot_{ACCOUNT_ID}.json")
snapshot_path.write_text(json.dumps(api_snapshot, ensure_ascii=False, indent=2), encoding="utf-8")

display(Markdown(f"Saved snapshot: `{snapshot_path}`"))
show_json("api_snapshot", api_snapshot)